# The algebroid declaration system

Working on an algebroid means **declaring** which structure you assume
(PDF item 10; papers: `pre-metric-bourbaki.pdf`, `metric-connection.pdf`).
What a declaration licenses fires; what it does not stays inert, and proofs
fail honestly. This notebook shows:

1. **Comparative table** — the same 4 statements under 5 declaration sets
2. **A multi-step proof** — one declared axiom + five definitional rules
3. **Failure triptych** — honest failure, a missing-declaration finder,
   and a genuinely false identity


In [1]:
from jacopy.core.registry import PropertyRegistry
from jacopy.core.expr import Product, Sum, Neg, Integer
from jacopy.algebra.derivation import Act
from jacopy.algebra.lie_bracket_vf import LieBracketVF
from jacopy.core.pairing import Pairing
from jacopy.central.objects import functions
from jacopy.central.algebroid import (
    algebroid, algebroid_engine, jacobiator, locality_term, DECLARATIONS,
)
from jacopy.proof.strategies import ExpandAndSimplify, ProofFailure

reg = PropertyRegistry()
f, g = functions("f g", registry=reg)


def try_prove(alg, lhs, rhs):
    try:
        ch = ExpandAndSimplify().prove(
            lhs, rhs, registry=reg, engine=algebroid_engine(alg, registry=reg)
        )
        return f"CLOSES ({len(ch.steps)})"
    except ProofFailure:
        return "-"


def show(chain, width=78):
    for i, s in enumerate(chain.steps, 1):
        print(f"  step {i:2d}: {s.rule[:66]}")
        print(f"      before: {s.before._repr_inner()[:width]}")
        print(f"      after : {s.after._repr_inner()[:width]}")
    print(f"  -- {len(chain.steps)} steps --")

## 1. Comparative table — same statements, five assumption sets

Each statement closes exactly in the sets containing its axiom; everywhere
else the shapes stay inert (`-`). Note the statements are the *same nodes*
in every column — assumptions live in the context/engine, never in the
expression.

In [2]:
sets = [
    ("(none)",         ()),
    ("almost-leibniz", ("almost-leibniz",)),
    ("local",          ("local",)),
    ("pre-leibniz",    ("pre-leibniz",)),
    ("leibniz",        ("leibniz",)),
]

def statements(alg):
    u, v, w = alg.sections("u v w")
    return [
        ("[u,fv] = ρ(u)(f)v + f[u,v]",
         alg.bracket(u, Product(f, v)),
         Sum(Product(Act(alg.anchor(u), f), v),
             Product(f, alg.bracket(u, v)))),
        ("[fu,v] = −ρ(v)(f)u + f[u,v] + L(Df,u,v)",
         alg.bracket(Product(f, u), v),
         Sum(Neg(Product(Act(alg.anchor(v), f), u)),
             Product(f, alg.bracket(u, v)),
             locality_term(alg, f, u, v))),
        ("ρ([u,v])(f) = [ρu,ρv](f)",
         Act(alg.anchor(alg.bracket(u, v)), f),
         Act(LieBracketVF(alg.anchor(u), alg.anchor(v)), f)),
        ("J(u,v,w) = 0",
         jacobiator(alg, u, v, w),
         Integer(0)),
    ]

names = [s[0] for s in statements(algebroid("E"))]
rows = {}
for set_name, decl in sets:
    alg = algebroid("E", declare=decl)
    rows[set_name] = [try_prove(alg, l, r) for _, l, r in statements(alg)]

w0 = max(len(n) for n in names) + 2
print(" " * w0 + " | ".join(f"{n:>14}" for n, _ in sets))
print("-" * (w0 + 17 * len(sets)))
for i, n in enumerate(names):
    print(f"{n:<{w0}}" + " | ".join(f"{rows[s][i]:>14}" for s, _ in sets))

                                                 (none) | almost-leibniz |          local |    pre-leibniz |        leibniz
------------------------------------------------------------------------------------------------------------------------------
[u,fv] = ρ(u)(f)v + f[u,v]                            - |     CLOSES (2) |     CLOSES (2) |     CLOSES (2) |     CLOSES (2)
[fu,v] = −ρ(v)(f)u + f[u,v] + L(Df,u,v)               - |              - |     CLOSES (2) |              - |              -
ρ([u,v])(f) = [ρu,ρv](f)                              - |              - |              - |     CLOSES (2) |              -
J(u,v,w) = 0                                          - |              - |              - |              - |     CLOSES (2)


## 2. A multi-step proof (declared: `leibniz`)

`⟨Df, [u, g·v]⟩ = ρ(u)(g)·ρ(v)(f) + g·ρ([u,v])(f)` — one declared axiom
(right-Leibniz) orchestrated with five definitional rules (coboundary,
anchor linearity ×3, operator linearity, scalar action). The step list
doubles as the **assumption record**: exactly one step is a declared
axiom, and it names itself.

In [3]:
E = algebroid("E", declare=("leibniz",))
u, v, w = E.sections("u v w")

lhs = Pairing(E.D(f), E.bracket(u, Product(g, v)))
rhs = Sum(
    Product(Act(E.anchor(u), g), Act(E.anchor(v), f)),
    Product(g, Act(E.anchor(E.bracket(u, v)), f)),
)
chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=algebroid_engine(E, registry=reg)
)
show(chain)

axioms_used = sorted({s.rule for s in chain.steps if "(E)" in s.rule})
print("\nassumptions used:", axioms_used)

  step  1: right-Leibniz (E): [u, fv] = ρ(u)(f)v + f[u,v]
      before: [u,(g * v)]_E
      after : ((ρ(u)(g) * v) + (g * [u,v]_E))
  step  2: coboundary (E): ⟨Df, u⟩ = ρ(u)(f)
      before: ⟨Df, ((ρ(u)(g) * v) + (g * [u,v]_E))⟩
      after : ρ(((ρ(u)(g) * v) + (g * [u,v]_E)))(f)
  step  3: anchor linearity: ρ(fu + v) = f·ρ(u) + ρ(v)
      before: ρ(((ρ(u)(g) * v) + (g * [u,v]_E)))
      after : (ρ((ρ(u)(g) * v)) + ρ((g * [u,v]_E)))
  step  4: anchor linearity: ρ(fu + v) = f·ρ(u) + ρ(v)
      before: ρ((ρ(u)(g) * v))
      after : (ρ(u)(g) * ρ(v))
  step  5: anchor linearity: ρ(fu + v) = f·ρ(u) + ρ(v)
      before: ρ((g * [u,v]_E))
      after : (g * ρ([u,v]_E))
  step  6: Act linearity: (A + B)(x) = A(x) + B(x)
      before: ((ρ(u)(g) * ρ(v)) + (g * ρ([u,v]_E)))(f)
      after : ((ρ(u)(g) * ρ(v))(f) + (g * ρ([u,v]_E))(f))
  step  7: product-rule
      before: (((ρ(u)(g) * ρ(v))(f) + (g * ρ([u,v]_E))(f)) + (-((ρ(u)(g) * ρ(v)(f)) + (g * ρ
      after : ((ρ(u)(g)(ρ(v)(f)) + g(ρ([u,v]_E)(

## 3a. Honest failure — the anchor-morphism statement under `leibniz`

Every Leibniz algebroid IS pre-Leibniz (the Phase 3.D theorem), but the
`leibniz` level deliberately does not hand the morphism out: a derivable
property is never silently assumed. With the plain engine this fails with
the untouched opaque residual — the theorem must be *derived and cited*
(next section).

In [4]:
lhs = Act(E.anchor(E.bracket(u, v)), f)
rhs = Act(LieBracketVF(E.anchor(u), E.anchor(v)), f)
try:
    ExpandAndSimplify().prove(
        lhs, rhs, registry=reg, engine=algebroid_engine(E, registry=reg)
    )
except ProofFailure as e:
    print("ProofFailure:")
    print(" ", str(e).splitlines()[0])

ProofFailure:
  ExpandAndSimplify left residual ((-[ρ(u),ρ(v)]_VF(f)) + ρ([u,v]_E)(f)) when proving ρ([u,v]_E)(f) == [ρ(u),ρ(v)]_VF(f)


## 3a′. The flagship theorem (Phase 3.D): derive, then cite

`right-Leibniz + Leibniz-Jacobi ⇒ ρ([u,v]) = [ρ(u), ρ(v)]` — the classical
auxiliary-section argument, fully mechanized: expand `J(u,v,f·w) = 0`
against a generic function `f` and a generic auxiliary section `w`, cancel
the generic section, recognize the commutator. The chain records both
declared axioms and the two generator-agreement inferences explicitly.

In [5]:
from jacopy.central.algebroid import prove_anchor_morphism
from jacopy.central.tangent.lie_bracket import LieBracketActionDefinition
from jacopy.proof.theorems import TheoremBook, cite

chain, thm = prove_anchor_morphism(E, u, v, f, registry=reg)
print(thm.statement)
print("from axioms:", thm.from_axioms)
print()
for i, s in enumerate(chain.steps, 1):
    sub = f"  [+{len(s.children)} sub-steps]" if s.children else ""
    print(f"  step {i}: {s.rule}{sub}")

ρ([u,v]) = [ρ(u), ρ(v)] on E (every Leibniz algebroid is pre-Leibniz)
from axioms: ('right-Leibniz (E)', 'Leibniz-Jacobi (E)')

  step 1: Leibniz-Jacobi (E): J(u,v,w) = 0
  step 2: expand J(u,v,f·w) by definitions + declared right-Leibniz  [+11 sub-steps]
  step 3: f·J(u,v,w) = 0 (declared Leibniz-Jacobi)  [+2 sub-steps]
  step 4: collect: J(u,v,f·w) − f·J(u,v,w) = A·w  [+3 sub-steps]
  step 5: agreement on generators (w)
  step 6: solve A = 0 for ρ([u,v])(f)
  step 7: Lie bracket definition: [ρ(u), ρ(v)](f) = ρ(u)(ρ(v)(f)) − ρ(v)(ρ(u)(f))  [+2 sub-steps]
  step 8: agreement on generators (f)


With the theorem **cited**, the statement of 3a now closes — and the
step is tagged `theorem`, not `axiom`; in foundational mode it carries the
whole derivation as children.

In [6]:
book = TheoremBook()
book.add(thm)
eng = algebroid_engine(E, registry=reg)
eng.register(LieBracketActionDefinition())
cite(eng, book, thm.name)

chain2 = ExpandAndSimplify().prove(
    Act(E.anchor(E.bracket(u, v)), f),
    Act(LieBracketVF(E.anchor(u), E.anchor(v)), f),
    registry=reg, engine=eng,
)
show(chain2)
print("\nprovenance:", [s.provenance_tag for s in chain2.steps])

  step  1: theorem anchor_morphism_E_u_v ⇒: ρ([u,v]) = [ρ(u), ρ(v)] on E (eve
      before: ρ([u,v]_E)
      after : [ρ(u),ρ(v)]_VF
  step  2: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
      before: [ρ(u),ρ(v)]_VF(f)
      after : (ρ(u)(ρ(v)(f)) + (-ρ(v)(ρ(u)(f))))
  step  3: Lie bracket definition: [X, Y](f) = X(Y(f)) - Y(X(f))
      before: [ρ(u),ρ(v)]_VF(f)
      after : (ρ(u)(ρ(v)(f)) + (-ρ(v)(ρ(u)(f))))
  step  4: simplify
      before: ((ρ(u)(ρ(v)(f)) + (-ρ(v)(ρ(u)(f)))) + (-(ρ(u)(ρ(v)(f)) + (-ρ(v)(ρ(u)(f))))))
      after : 0
  -- 4 steps --

provenance: ['theorem', 'axiom', 'axiom', None]


## 3b. The missing-declaration finder

Because assumptions are modular and instance-scoped, we can search the
assumption space mechanically: retry the statement with each candidate
axiom added.

In [7]:
def find_missing_declaration(alg, lhs_of, rhs_of):
    print(f"declared {sorted(alg.declarations)}:",
          try_prove(alg, lhs_of(alg), rhs_of(alg)))
    for d in DECLARATIONS:
        if alg.declares(d):
            continue
        cand = alg.with_declarations(d)
        r = try_prove(cand, lhs_of(cand), rhs_of(cand))
        print(f"  + {d:16s}: {r}")
        if r != "-":
            print(f"  => MISSING DECLARATION: {d}")
            return
    print("  => no single declaration closes it — "
          "the statement may simply be false")


find_missing_declaration(
    E,
    lambda a: Act(a.anchor(a.bracket(*a.sections("u v"))), f),
    lambda a: Act(LieBracketVF(a.anchor(a.sections("u v")[0]),
                               a.anchor(a.sections("u v")[1])), f),
)

declared ['jacobi', 'right-leibniz']: -
  + left-leibniz    : -
  + anchor-morphism : CLOSES (2)
  => MISSING DECLARATION: anchor-morphism


## 3c. A genuinely false identity

`[u, g·v] = g·[u,v]` (the `ρ(u)(g)·v` term dropped). This is not a
missing assumption — it is wrong. With EVERY declaration on, the residual
survives as a concrete term; and the finder reports that no declaration
helps. Two different failures, two different signatures.

In [8]:
E_all = algebroid("E", declare=("local", "jacobi", "anchor-morphism"))
u, v = E_all.sections("u v")
try:
    ExpandAndSimplify().prove(
        E_all.bracket(u, Product(g, v)),
        Product(g, E_all.bracket(u, v)),
        registry=reg, engine=algebroid_engine(E_all, registry=reg),
    )
except ProofFailure as e:
    print("ProofFailure (all declarations on):")
    print(" ", str(e).splitlines()[0])

print()
find_missing_declaration(
    algebroid("E"),
    lambda a: a.bracket(a.sections("u v")[0], Product(g, a.sections("u v")[1])),
    lambda a: Product(g, a.bracket(*a.sections("u v"))),
)

ProofFailure (all declarations on):
  ExpandAndSimplify left residual (v * ρ(u)(g)) when proving [u,(g * v)]_E == (g * [u,v]_E)

declared []: -
  + right-leibniz   : -
  + left-leibniz    : -
  + anchor-morphism : -
  + jacobi          : -
  => no single declaration closes it — the statement may simply be false
